In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------- 1. Load data ----------
df = pd.read_csv('student_placement_prediction_dataset_2026.csv')

# student_id sirf identifier hai, koi predictive value nahi -> drop
# salary_package_lpa target ka direct leak hai (Not Placed => 0 hamesha) -> drop
df = df.drop(columns=['student_id', 'salary_package_lpa'])


y = (df['placement_status'] == 'Placed').astype(int)   # Placed=1, Not Placed=0
X = df.drop(columns=['placement_status'])



In [44]:

cat_cols = X.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", cat_cols)

encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c])
    encoders[c] = le  


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)
log_acc = accuracy_score(y_test, log_pred)

print("\n===== LOGISTIC REGRESSION =====")
print(f"Accuracy: {log_acc:.4f}")
print(classification_report(y_test, log_pred, target_names=['Not Placed', 'Placed']))
print("Confusion Matrix:\n", confusion_matrix(y_test, log_pred))


rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print("\n===== RANDOM FOREST =====")
print(f"Accuracy: {rf_acc:.4f}")
print(classification_report(y_test, rf_pred, target_names=['Not Placed', 'Placed']))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))



Categorical columns: []

===== LOGISTIC REGRESSION =====
Accuracy: 0.5715
              precision    recall  f1-score   support

  Not Placed       0.55      0.33      0.41      9108
      Placed       0.58      0.77      0.66     10892

    accuracy                           0.57     20000
   macro avg       0.56      0.55      0.54     20000
weighted avg       0.57      0.57      0.55     20000

Confusion Matrix:
 [[3001 6107]
 [2463 8429]]

===== RANDOM FOREST =====
Accuracy: 0.5593
              precision    recall  f1-score   support

  Not Placed       0.53      0.33      0.40      9108
      Placed       0.57      0.75      0.65     10892

    accuracy                           0.56     20000
   macro avg       0.55      0.54      0.53     20000
weighted avg       0.55      0.56      0.54     20000

Confusion Matrix:
 [[2991 6117]
 [2698 8194]]


In [42]:
# ---------- 7. Feature importance (Random Forest se) ----------
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)
print("\n===== FEATURE IMPORTANCE (Random Forest) =====")
print(importances)


===== FEATURE IMPORTANCE (Random Forest) =====
mock_interview_score         0.067381
coding_skill_score           0.067300
logical_reasoning_score      0.066536
aptitude_score               0.066368
communication_skill_score    0.066074
leadership_score             0.065382
extracurricular_score        0.064827
linkedin_connections         0.064050
cgpa                         0.063151
attendance_percentage        0.062477
study_hours_per_day          0.056180
sleep_hours                  0.055973
github_repos                 0.034220
age                          0.030243
certifications_count         0.027991
projects_count               0.027791
branch                       0.025318
hackathons_participated      0.020647
internships_count            0.020633
college_tier                 0.015898
backlogs                     0.012944
gender                       0.009355
volunteer_experience         0.009261
dtype: float64


In [45]:
# ---------- 8. Final comparison ----------
print("\n===== SUMMARY =====")
print(f"Logistic Regression Accuracy: {log_acc:.4f}")
print(f"Random Forest Accuracy:       {rf_acc:.4f}")


===== SUMMARY =====
Logistic Regression Accuracy: 0.5715
Random Forest Accuracy:       0.5593
